# GNSS in Lunar Orbit
[1] S. Bhamidipati, T. Mina, and G. Gao, ‘Design Considerations of a Lunar Navigation Satellite System with Time-Transfer from Earth-GPS’, presented at the Proceedings of the 34th International Technical Meeting of the Satellite Division of The Institute of Navigation (ION GNSS+ 2021), Sep. 2021, pp. 950–965. doi: 10.33012/2021.18021.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

%load_ext autoreload
%autoreload 2
import pylupnt as pnt

flag_plot = True

In [ ]:
# Epoch(TAI)
t0_tai = pnt.convert_time(pnt.gregorian2time(2025, 11, 9, 0, 0, 0), pnt.UTC, pnt.TAI)
N_sc = 3

# Classical orbital elements (a, e, i, W, w, M) [km, -, rad, rad, rad, rad]
sma = [6541.4, 7500, 9750.5]  # [km] Semi-major axis
ecc = [0.6, 0.05, 0.7]  # [-] Eccentricity
inc = np.deg2rad([56.2, 40, 65.5])  # [rad] Inclination
asc = np.deg2rad([0, 0, 0])  # [rad] Right ascension of the ascending node
aop = np.deg2rad([90, 90, 90])  # [rad] Argument of periapsis
man = np.deg2rad([0, 0, 0])  # [rad] Mean anomaly

coe_op = np.zeros((N_sc, 6))
for i in range(N_sc):
    coe_op[i] = np.array([sma[i], ecc[i], inc[i], asc[i], aop[i], man[i]])

rv0_m2sc_op = pnt.classical2cart(coe_op, pnt.GM_MOON)
rv0_m2sc_ci = pnt.convert_frame(t0_tai, rv0_m2sc_op, pnt.MOON_OP, pnt.MOON_CI)

# Time
period = 2 * np.pi * np.sqrt(np.power(coe_op[0, 0], 3) / pnt.GM_MOON)  # [s] Orbital period
dt = 2 * pnt.SECS_MINUTE  # [s] Simulation time step
dt_prop = 15  # [s] Propagation time step
tf = 2 * 31 * pnt.SECS_DAY  # [s] Simulation final time
N_t = int(tf / dt)  # [-] Number of time steps
tspan = np.linspace(0, tf, N_t)  # [s] Time since first epoch
tspan_h = tspan / pnt.SECS_HOUR  # [h] Time since first epoch
tspan_day = tspan / pnt.SECS_DAY  # [day] Time since first epoch
t_tai = t0_tai + tspan  # [s] Epochs (TAI)

# Dynamics
dyn = pnt.NBodyDynamics()
dyn.add_body(pnt.Body.Moon(5, 5))
dyn.add_body(pnt.Body.Earth())
dyn.add_body(pnt.Body.Sun())
dyn.set_frame(pnt.MOON_CI)
dyn.set_time_step(dt_prop)

# Propagation
# rv_from2to_frame [km, km/s] (x, y, z, vx, vy, vz)
rv_m2sc_ci = np.zeros((N_sc, N_t, 6))
rv_m2sc_pa = np.zeros((N_sc, N_t, 6))
for i in range(N_sc):
    rv_m2sc_ci[i] = dyn.propagate(rv0_m2sc_ci[i], t0_tai, t_tai)
    rv_m2sc_pa[i] = pnt.convert_frame(t_tai, rv_m2sc_ci[i], pnt.MOON_CI, pnt.MOON_PA)

# Earth and Sun
rv_m2e_ci = pnt.get_body_pos_vel(t_tai, pnt.MOON, pnt.EARTH, pnt.MOON_CI)
rv_m2e_pa = pnt.convert_frame(t_tai, rv_m2e_ci, pnt.MOON_CI, pnt.MOON_PA)
rv_m2s_ci = pnt.get_body_pos_vel(t_tai, pnt.MOON, pnt.SUN, pnt.MOON_CI)
rv_m2s_pa = pnt.convert_frame(t_tai, rv_m2s_ci, pnt.MOON_CI, pnt.MOON_PA)
rv_e2s_eci = pnt.get_body_pos_vel(t_tai, pnt.EARTH, pnt.SUN, pnt.ECI)

In [ ]:
if flag_plot:
    fig = go.Figure()
    pnt.plot.plot_orbits(fig, rv_m2sc_ci[:, ::10])
    pnt.plot.plot_body(
        fig,
        pnt.MOON,
        size_factor=2,
        alpha=0.5,
    )
    pnt.plot.set_view(fig, -80, 20, 2.5)
    fig.update_layout(showlegend=True, width=400, height=400)
    fig.show()

In [ ]:
tles = pnt.TLE.from_file("gps")
N_gps = len(tles)
coe_gps_eci = np.zeros((N_gps, N_t, 6))
rv_gps_eci = np.zeros((N_gps, N_t, 6))
rv_gps_ci = np.zeros((N_gps, N_t, 6))
dyn_gps = pnt.KeplerianDynamics(pnt.GM_EARTH)
prns = np.zeros(N_gps, dtype=int)
for i in range(N_gps):
    coe0_gps = pnt.tle2classical(tles[i], pnt.GM_EARTH)
    coe_gps_eci[i] = dyn_gps.propagate(coe0_gps, tles[i].epoch_tai, t_tai)
    rv_gps_eci[i] = pnt.classical2cart(coe_gps_eci[i], pnt.GM_EARTH)
    rv_gps_ci[i] = pnt.convert_frame(t_tai, rv_gps_eci[i], pnt.ECI, pnt.MOON_CI)
    prns[i] = tles[i].prn

In [ ]:
# Read data from file
import os

df = pd.read_csv(pnt.find_file('gps_table.csv'))
gps_antenna_names = df.set_index("PRN")["ACE_File"].to_dict()
gps_antennas = {k: pnt.Antenna(v) for k, v in gps_antenna_names.items()}

rx_antenna = pnt.Antenna("moongpsr")

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(10,3))
pnt.plot.plot_antenna_gain_patter_2D(antenna=gps_antennas[2], ax=axs[0])
pnt.plot.plot_antenna_gain_patter_2D(antenna=rx_antenna, ax=axs[1])
plt.xlim(-5,5)
plt.ylim(10,15)
plt.tight_layout()
plt.legend()

In [ ]:
e_gps2e = pnt.normalize(-rv_gps_eci[:, :, :3])
e_gps2s = pnt.normalize(rv_e2s_eci[None, :, :3] - rv_gps_eci[:, :, :3])

ez_gps = e_gps2e
ey_gps = pnt.cross_norm(e_gps2e, e_gps2s)
ex_gps = pnt.cross_norm(ey_gps, ez_gps)

In [ ]:
t = 0
if flag_plot:
    tickvals = np.arange(-30, 31, 10) * 1e3
    fig = go.Figure()
    pnt.plot.plot_orbits(fig, rv_gps_eci[:, :int(pnt.SECS_DAY / dt):10], t=t, color="lightgray")
    pnt.plot.plot_body(fig, pnt.EARTH, size_factor=5)
    for i in range(N_gps):
        pnt.plot.plot_frame(
            fig, rv_gps_eci[i, t, :3], np.vstack((ex_gps[i, t], ey_gps[i, t], ez_gps[i, t])), length=pnt.R_EARTH, width=5, tip=5
        )
    pnt.plot.plot_arrow3(fig, np.zeros(3), pnt.normalize(rv_e2s_eci[t]), length=3 * pnt.R_EARTH, width=5, color="orange", tip=10)
    pnt.plot.set_view(fig, 50, 20, 2.5)
    # fig.update_layout(width=400, height=400)
    fig.show()

In [ ]:
def compute_visibility(
    r1: np.ndarray, r2: np.ndarray, R_body: float, r_body: np.ndarray = None
) -> np.ndarray:
    if r_body is None:
        r_body = np.zeros(3)
    r = r2 - r1
    r_norm = np.linalg.norm(r, axis=1)
    r1body = r1 - r_body
    r1body_norm = np.linalg.norm(r1body, axis=1)
    dot = np.einsum("ij,ij->i", -r1body, r)
    theta1 = np.arccos(np.clip(dot / r_norm / r1body_norm, -1, 1))
    theta2 = np.arcsin(np.clip(R_body / r1body_norm, -1, 1))
    visibility = np.ones(len(r1), dtype=bool)
    visibility[(theta1 < theta2) & (r_norm > r1body_norm)] = False
    return visibility

In [ ]:
# Visibility
vis_sc2sc = np.ones((N_sc, N_sc, N_t), dtype=bool)
vis_sc2gps = np.ones((N_sc, N_gps, N_t), dtype=bool)
dist_sc2sc = np.zeros((N_sc, N_sc, N_t))
dist_sc2gps = np.zeros((N_sc, N_gps, N_t))
phi_gps2sc = np.zeros((N_sc, N_gps, N_t))
phi_sc2gps = np.zeros((N_sc, N_gps, N_t))
theta_gps2sc = np.zeros((N_sc, N_gps, N_t))
G_tx = np.zeros((N_sc, N_gps, N_t))
G_rx = np.zeros((N_sc, N_gps, N_t))

for i in range(N_sc):
    for j in range(i + 1, N_sc):
        vis_sc2sc[i, j] &= compute_visibility(
            rv_m2sc_ci[i, :, :3], rv_m2sc_ci[j, :, :3], pnt.R_MOON
        )
        vis_sc2sc[j, i] &= vis_sc2sc[i, j]
        dist_sc2sc[i, j] = np.linalg.norm(
            rv_m2sc_ci[i, :, :3] - rv_m2sc_ci[j, :, :3], axis=1
        )
        dist_sc2sc[j, i] = dist_sc2sc[i, j]

    for j in range(N_gps):
        vis_sc2gps[i, j] &= compute_visibility(
            rv_m2sc_ci[i, :, :3], rv_gps_ci[j, :, :3], pnt.R_MOON
        )
        vis_sc2gps[i, j] &= compute_visibility(
            rv_m2sc_ci[i, :, :3], rv_gps_ci[j, :, :3], pnt.R_EARTH, rv_m2e_ci[:, :3]
        )
        dist_sc2gps[i, j] = np.linalg.norm(
            rv_m2sc_ci[i, :, :3] - rv_gps_ci[j, :, :3], axis=1
        )

        u_gps2sc = pnt.normalize(rv_m2sc_ci[i, :, :3] - rv_gps_ci[j, :, :3])
        phi_gps2sc[i, j] = np.arccos(
            np.clip(np.sum(u_gps2sc * ez_gps[j], axis=-1), -1, 1)
        )
        theta_gps2sc[i, j] = np.arctan2(
            np.sum(u_gps2sc * ey_gps[j], axis=-1), np.sum(u_gps2sc * ex_gps[j], axis=-1)
        )

        u_sc2gps = -u_gps2sc
        u_sc2e = pnt.normalize(rv_m2e_ci[:, :3] - rv_m2sc_ci[i, :, :3])
        phi_sc2gps[i, j] = np.arccos(np.clip(np.sum(u_sc2e * u_sc2gps, axis=-1), -1, 1))

        G_tx[i, j] = gps_antennas[prns[j]].compute_gain(theta_gps2sc[i, j], phi_gps2sc[i, j])
        G_rx[i, j] = rx_antenna.compute_gain(0, phi_sc2gps[i, j])

        G_tx[i,j][~vis_sc2gps[i, j]] = np. nan
        G_rx[i,j][~vis_sc2gps[i, j]] = np. nan

    vis_sc2sc[i, i, :] = False

In [ ]:
# Link budget
P_tx = 15  # [dBW] Transmitter power
freq = 1575.42e6  # [Hz] GPS L1 frequency
L_ad = 0.0  # [dB] A/D converter loss
L_atm = 0.0  # [dB] Atmospheric loss
Nf = 2  # [dB] Noise figure
Tsys = 113  # [K] System noise temperature
# [dB] Free space loss
L_fs = 20 * np.log10((4 * np.pi * dist_sc2gps) / (pnt.C / freq))
# [dB] Transmitter antenna gain
kb_db = 10 * np.log10(1.38e-23)  # [dB] Boltzmann constant
CN0 = P_tx + G_tx + G_rx - L_atm - L_fs - L_ad - Nf - kb_db - 10 * np.log10(Tsys)
CN0_threshold = 15  # [dB-Hz] CN0 thresholds

B_dll = 0.5 # [Hz] Bandwidth
d = 0.3 # [chip] Chip duration
Ti = 20e-3 # [s] Coherent integration time
B_fe = 26e6 # [Hz] Front-end bandwidth
fc = 1.023e6  # [Hz] Spread code frequency 
Tc = 1 / fc  # [s] Spread code period 

# Error contributions
# Delay Lock Loop (DLL) error [m]
term_1 = (pnt.C * Tc) ** 2 * (B_dll * (1 - 0.5 * B_dll * Ti)) / (2 * CN0)
if d >= np.pi / (Tc * B_fe):
    sigma_dll_sq = term_1 * d
elif np.pi / (Tc * B_fe) > d > 1 / (Tc * B_fe):
    term_2 = (1 / (Tc * B_fe)) + ((Tc * B_fe) / (np.pi - 1)) * (
        d - (1 / (Tc * B_fe))
    ) ** 2
    term_3 = 1 + (2 / (Ti * CN0 * (2 - d)))
    sigma_dll_sq = term_1 * term_2 * term_3
else:  # d <= 1 / (T_c * B_fe)
    term_2 = 1 / (Tc * B_fe)
    term_3 = 1 + (1 / (Ti * CN0))
    sigma_dll_sq = term_1 * term_2 * term_3

sigma_gps = 0
sigma_rho = np.sqrt((pnt.C * Tc) ** 2 * sigma_dll_sq + sigma_gps ** 2)

In [ ]:
figsize = (10,3)
plt.rcParams.update({"font.size": 10})
if flag_plot:
    plt.figure(figsize=figsize)
    ylims = [5, 40]
    i = 0
    j = prns.tolist().index(7)
    y = CN0[i, j]
    plt.plot(tspan_day, y, "-o", linewidth=1, markersize=2)
    plt.axhline(CN0_threshold, color="black", linestyle="--", label="Threshold")
    plt.xlim(tspan_day[0], tspan_day[-1])
    plt.ylim(ylims)
    plt.xlabel("Time [days]")
    plt.ylabel("C/N0 [dB-Hz]")
    plt.grid()
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=figsize)
    for i in range(1):
        y = np.sum(CN0[i] > CN0_threshold, axis=0)
        plt.plot(tspan_day, y.T, "-o", label=f"Satellite {i+1}", markersize=2)
    plt.xlim(tspan_day[0], tspan_day[-1])
    plt.xlabel("Time [days]")
    plt.ylabel("Visible GPS satellites")
    plt.legend(loc="upper right")
    ylim = 14
    plt.yticks(np.arange(0, ylim + 1, 2))
    plt.ylim(0, ylim)
    plt.grid()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=figsize)
    idx1 = int(5.8 * pnt.SECS_DAY // dt)
    idx2 = int(7.5 * pnt.SECS_DAY // dt)
    for i in range(3):
        y = np.sum(CN0[i] > CN0_threshold, axis=0)
        plt.plot(tspan_day[idx1:idx2], y.T[idx1:idx2], "-o", label=f"Satellite {i+1}", markersize=2)
    plt.xlim(tspan_day[idx1], tspan_day[idx2])
    plt.xlabel("Time [days]")
    plt.ylabel("Visible GPS satellites")
    plt.legend(loc="upper right")
    plt.yticks(np.arange(0, ylim + 1, 2))
    plt.ylim(0, ylim)
    plt.grid()
    plt.tight_layout()
    plt.show()